# Calcul des embeddings

Ce notebook reconstruit de manière pédagogique la logique du script `src/compute_embeddings.py`.

L'objectif est de comprendre clairement **ce que fait chaque étape**, et pas seulement d'exécuter le pipeline "en boîte noire".

À la fin du notebook, on aura :
- chargé la configuration du projet ;
- sélectionné les splits utilisés pour construire la galerie ;
- regroupé les images par style ;
- chargé l'encodeur entraîné ;
- calculé un embedding par image ;
- sauvegardé les fichiers `.npy`, le bundle `.npz` et le manifeste JSON.


## 1) Préparer l'environnement du notebook

Comme ce notebook vit dans `art-xplain/notebooks/`, on ajoute la racine du projet à `sys.path` pour pouvoir importer `src.compute_embeddings` et `src.utils` proprement.


In [ ]:
from pathlib import Path
import sys

# Racine du projet Python `art-xplain/`.
PROJECT_ROOT = Path.cwd().resolve().parents[0]

# Si on exécute le notebook depuis `art-xplain/notebooks`, alors `PROJECT_ROOT`
# pointe vers `art-xplain/`, qui contient le package `src/`.
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print(f"src disponible ? {(PROJECT_ROOT / 'src').exists()}")


## 2) Importer les fonctions du script source

Au lieu de réécrire toute la logique à la main, on réutilise les mêmes helpers que le script `compute_embeddings.py`. Cela garantit que le notebook reste aligné avec le code réellement utilisé par l'application.


In [ ]:
import json
import numpy as np
import pandas as pd
import tensorflow as tf

from src.utils import ensure_dir, load_config, relativize_project_path, resolve_project_path
from src.compute_embeddings import (
    _collect_images_by_class,
    _compute_embedding,
    _resolve_dataset_sources,
    _sha1_of_array,
)

print("Imports OK")


## 3) Charger la configuration du projet

Le script principal lit `config/config.yaml`. Ici, on fait pareil.

Point important :
- si `paths.dataset_root` est défini, il est prioritaire ;
- sinon, le notebook utilisera `dataset.embedding_splits` pour choisir les splits à encoder.


In [ ]:
cfg = load_config()

# Résout la ou les sources d'images selon la configuration actuelle.
dataset_roots, dataset_source = _resolve_dataset_sources(cfg)
embeddings_root = resolve_project_path(cfg['paths']['embeddings_root'])
models_root = resolve_project_path(cfg['paths']['models_root'])
img_size = int(cfg['model']['img_size'])

print('Mode de sélection des données :', dataset_source)
print('Sources utilisées :')
for root in dataset_roots:
    print(' -', root.resolve())
print('Dossier de sortie embeddings :', embeddings_root.resolve())
print('Dossier des modèles         :', models_root.resolve())
print('Taille image                :', img_size)


## 4) Vérifier le paramètre `embedding_splits`

Dans ce projet, on peut maintenant piloter la construction de la galerie directement dans le `config.yaml`.

Exemples utiles :
- `['train', 'val']` : recommandé pour l'application de retrieval ;
- `['train', 'val', 'test']` : pour encoder toute la collection disponible.


In [ ]:
dataset_cfg = cfg.get('dataset', {})
print('dataset.embedding_splits =', dataset_cfg.get('embedding_splits', ['train', 'val']))
print('paths.dataset_root       =', cfg.get('paths', {}).get('dataset_root', '<non défini>'))


## 5) Regrouper les images par style

Le script ne traite pas les splits séparément jusqu'au bout.
Il commence par **fusionner les images par nom de classe**.

Exemple :
- `train/impressionism/...`
- `val/impressionism/...`

seront regroupés dans une seule entrée `impressionism`.


In [ ]:
images_by_class = _collect_images_by_class(dataset_roots)

if not images_by_class:
    raise ValueError('Aucune image trouvée dans les sources configurées.')

summary_rows = []
for class_name, image_files in images_by_class.items():
    summary_rows.append({
        'style': class_name,
        'num_images': len(image_files),
        'exemple_fichier': image_files[0].name if image_files else '<vide>',
    })

summary_df = pd.DataFrame(summary_rows).sort_values('num_images', ascending=False).reset_index(drop=True)
print(f"Nombre de styles détectés : {len(summary_df)}")
summary_df.head(10)


## 6) Inspecter une classe et quelques chemins d'images

Avant de lancer un calcul potentiellement long, c'est une bonne pratique de vérifier que les chemins et les classes ont bien l'air cohérents.


In [ ]:
first_class = summary_df.iloc[0]['style']
first_images = images_by_class[first_class][:5]

print('Classe inspectée :', first_class)
for img_path in first_images:
    print(' -', img_path)


## 7) Charger l'encodeur entraîné

On charge exactement le même modèle que celui utilisé ensuite par le moteur de retrieval.

Le fichier attendu est `models/encoder.keras`.


In [ ]:
encoder_path = models_root / 'encoder.keras'

if not encoder_path.exists():
    raise FileNotFoundError(f'encoder.keras introuvable : {encoder_path}')

encoder = tf.keras.models.load_model(
    encoder_path,
    compile=False,
    safe_mode=False,
)

print('Modèle chargé depuis :', encoder_path.resolve())
print('Type du modèle       :', type(encoder).__name__)


## 8) Tester le pipeline sur une seule image

Avant de calculer tous les embeddings, on vérifie qu'une image passe bien dans l'encodeur et que la sortie a la forme attendue.


In [ ]:
sample_image = next(iter(images_by_class.values()))[0]
sample_embedding = _compute_embedding(encoder, sample_image, img_size)

print('Image test          :', sample_image)
print('Shape embedding     :', sample_embedding.shape)
print('Dtype embedding     :', sample_embedding.dtype)
print('Premières valeurs   :', sample_embedding[:10])


## 9) Calculer les embeddings pour toute la galerie

On reproduit maintenant la logique principale de `compute_embeddings.py` :
- parcourir les classes ;
- calculer un embedding pour chaque image ;
- construire en parallèle les labels, les noms de classes et la liste des fichiers.

Cette cellule peut prendre un peu de temps selon la taille du dataset.


In [ ]:
vectors_list = []
labels_list = []
filenames_list = []
classnames_list = []
skipped_files = []

print(f"Nombre total de styles à traiter : {len(images_by_class)}")

for class_name, image_files in images_by_class.items():
    if not image_files:
        print(f"[WARN] Classe vide ignorée : {class_name}")
        continue

    class_idx = len(classnames_list)
    classnames_list.append(class_name)

    valid_count = 0
    for img_path in image_files:
        try:
            vec = _compute_embedding(encoder, img_path, img_size)
            vectors_list.append(vec)
            labels_list.append(class_idx)
            filenames_list.append(relativize_project_path(img_path))
            valid_count += 1
        except Exception as exc:
            skipped_files.append({
                'file': str(img_path),
                'error': str(exc),
            })

    print(f"- {class_name}: {valid_count}/{len(image_files)} embeddings calculés")

print('\nCalcul terminé')
print('Embeddings valides :', len(vectors_list))
print('Fichiers ignorés   :', len(skipped_files))


## 10) Convertir les listes Python en tableaux NumPy

Pendant le calcul, on utilise des listes Python car elles sont faciles à alimenter au fil de l'eau.

À la fin, on convertit tout en tableaux NumPy homogènes pour :
- faire des vérifications de cohérence ;
- sauvegarder proprement les résultats ;
- faciliter l'utilisation future dans `retrieval.py` et `visualization_umap.py`.


In [ ]:
if not vectors_list:
    raise ValueError("Aucun embedding n'a pu être calculé.")

vectors = np.stack(vectors_list).astype(np.float32)
labels = np.asarray(labels_list, dtype=np.int64)
filenames = np.asarray(filenames_list, dtype=object)
classnames = np.asarray(classnames_list, dtype=object)

print('vectors.shape   =', vectors.shape)
print('labels.shape    =', labels.shape)
print('filenames.shape =', filenames.shape)
print('classnames      =', list(classnames))


## 11) Vérifier la cohérence des résultats

Chaque embedding doit correspondre à :
- un label ;
- un chemin de fichier ;
- une classe valide dans `classnames`.

Ces contrôles évitent de produire un export incohérent.


In [ ]:
n_samples = vectors.shape[0]

if not (n_samples == len(labels) == len(filenames)):
    raise RuntimeError(
        'Incohérence de cardinalité après calcul des embeddings.\n'
        f' - vectors   : {n_samples}\n'
        f' - labels    : {len(labels)}\n'
        f' - filenames : {len(filenames)}'
    )

if labels.size > 0:
    min_label = int(labels.min())
    max_label = int(labels.max())
    if min_label < 0 or max_label >= len(classnames):
        raise RuntimeError(
            'Incohérence entre labels et classnames.\n'
            f' - min_label   : {min_label}\n'
            f' - max_label   : {max_label}\n'
            f' - num_classes : {len(classnames)}'
        )

print('Contrôles OK')
print("Nombre d'échantillons :", n_samples)
print('Dimension embedding    :', vectors.shape[1])


## 12) Sauvegarder l'export complet

On écrit maintenant les mêmes artefacts que le script `compute_embeddings.py` :
- `vectors.npy`
- `labels.npy`
- `filenames.npy`
- `classnames.npy`
- `embeddings_bundle.npz`
- `embeddings_manifest.json`


In [ ]:
ensure_dir(embeddings_root)

bundle_path = embeddings_root / 'embeddings_bundle.npz'
np.savez_compressed(
    bundle_path,
    vectors=vectors,
    labels=labels,
    filenames=filenames,
    classnames=classnames,
)

np.save(embeddings_root / 'vectors.npy', vectors)
np.save(embeddings_root / 'labels.npy', labels)
np.save(embeddings_root / 'filenames.npy', filenames)
np.save(embeddings_root / 'classnames.npy', classnames)

manifest = {
    'num_samples': int(n_samples),
    'embedding_dim': int(vectors.shape[1]),
    'num_classes': int(len(classnames)),
    'img_size': int(img_size),
    'dataset_roots': [str(path.resolve()) for path in dataset_roots],
    'dataset_source': dataset_source,
    'embeddings_root': str(embeddings_root.resolve()),
    'encoder_path': str(encoder_path.resolve()),
    'vectors_sha1': _sha1_of_array(vectors),
    'labels_sha1': _sha1_of_array(labels),
    'filenames_sha1': _sha1_of_array(np.asarray([str(x) for x in filenames], dtype='<U2048')),
    'classnames_sha1': _sha1_of_array(np.asarray([str(x) for x in classnames], dtype='<U512')),
    'skipped_count': len(skipped_files),
    'skipped_files': skipped_files[:200],
}

manifest_path = embeddings_root / 'embeddings_manifest.json'
manifest_path.write_text(
    json.dumps(manifest, indent=2, ensure_ascii=False),
    encoding='utf-8',
)

print('Export terminé')
print(' - bundle   :', bundle_path.resolve())
print(' - manifest :', manifest_path.resolve())


## 13) Prévisualiser le manifeste produit

Cette dernière cellule permet de vérifier rapidement les métadonnées générées par l'export.


In [ ]:
manifest_preview = pd.Series(manifest)
manifest_preview


## 14) Ce qu'il faut retenir

Ce notebook montre que `compute_embeddings.py` fait essentiellement 4 choses :

1. il choisit les sources d'images à partir de la configuration ;
2. il transforme chaque image en vecteur via l'encodeur ;
3. il maintient la correspondance entre vecteurs, labels, fichiers et styles ;
4. il exporte un ensemble cohérent de fichiers réutilisables par le retrieval et l'UMAP.

Si l’on modifie `dataset.embedding_splits` dans `config.yaml`, cela change directement la galerie utilisée pour construire les embeddings.
